# Gemma 4 E2B - AI Router Model Fine-Tuning
## Google Colab Free Tier (T4 GPU, 16GB VRAM)

**Mục đích:** Fine-tune Gemma 4 E2B làm router model — hiểu intent của Sếp, trích xuất thông tin, tạo enriched prompt cho model lớn xử lý.

**Framework:** Unsloth + QLoRA
**Output:** GGUF model quantized Q4_K_M (~1.5GB) cho CPU inference
**Model upload:** Hugging Face Hub (diepxuan/dx-gemma4)

In [ ]:
# Cell 1: Install Dependencies
!pip install unsloth --quiet
!pip install "gguf>=0.10.0" --quiet
!pip install scikit-learn --quiet
!pip install huggingface_hub --quiet
print("Dependencies installed!")

In [ ]:
# Cell 2: Load Gemma 4 E2B Model
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/gemma-4-e2b-unsloth-bnb-4bit",
    max_seq_length=2048,
    dtype=None,
    load_in_4bit=True,
)
print("Model loaded successfully!")

In [ ]:
# Cell 3: Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.1,
    bias="none",
    use_gradient_checkpointing=True,
    random_state=42,
)
print("LoRA adapters configured!")

In [ ]:
# Cell 4: Setup Chat Template
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="gemma-3",
)

In [ ]:
# Cell 5: Load Dataset from GitHub (instead of Google Drive)
!git clone https://github.com/diepxuan/dx-gemma4.git /content/dataset-repo

import json
import os

def load_jsonl(filepath):
    dataset = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line)
            dataset.append({"text": data["text"]})
    return dataset

dataset_paths = [
    "/content/dataset-repo/scripts/dataset/training_data.jsonl",
    "/content/training_data.jsonl",
]

dataset = None
for path in dataset_paths:
    if os.path.exists(path):
        print(f"Loading dataset from: {path}")
        dataset = load_jsonl(path)
        break

if dataset is None:
    print("No dataset found, generating minimal synthetic data...")
    dataset = [
        {"text": f"<|user|>\nSếp cần viết API REST cho user management bằng FastAPI\n<|assistant|>\n{{\"task_type\":\"code_generation\",\"domain\":\"user_management\",\"framework\":\"FastAPI\",\"complexity\":\"medium\",\"enriched_prompt\":\"Bạn là senior backend developer. Tạo FastAPI RESTful API cho user management. Bao gồm: endpoints, models, error handling. Code production-ready.\",\"needs_search\":false,\"recommended_model\":\"coding-model\"}}<|end|>"},
    ] * 100

print(f"Loaded {len(dataset)} examples")

# Train/eval split
from sklearn.model_selection import train_test_split
train_data, eval_data = train_test_split(dataset, test_size=0.1, random_state=42)
print(f"Train: {len(train_data)}, Eval: {len(eval_data)}")

In [ ]:
# Cell 6: Create Hugging Face Dataset Format
from datasets import Dataset

train_ds = Dataset.from_list(train_data)
eval_ds = Dataset.from_list(eval_data)

print(f"Datasets created: {len(train_ds)} train, {len(eval_ds)} eval")

In [ ]:
# Cell 7: Training Configuration
from trl import SFTConfig
from unsloth import is_bfloat16_supported

training_args = SFTConfig(
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    optim="adamw_8bit",
    fp16=True,
    bf16=is_bfloat16_supported(),
    eval_strategy="steps",
    eval_steps=100,
    logging_steps=20,
    save_steps=200,
    save_total_limit=2,
    output_dir="/content/gemma4-router-output",
    report_to="none",
    gradient_checkpointing=True,
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    dataset_text_field="text",
    max_seq_length=2048,
    dataset_num_proc=2,
    packing=False,
)
print("Training args configured!")

In [ ]:
# Cell 8: Initialize SFTTrainer
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=training_args,
)
print("Trainer initialized!")

In [ ]:
# Cell 9: Train the Model
import traceback

print("Starting training...")
try:
    trainer_stats = trainer.train()
    print("Training complete!")

    # TrainOutput from TRL/Transformers exposes `global_step` and `metrics`.
    # It does not expose `training_steps`; using it caused an AttributeError
    # after training finished successfully.
    metrics = getattr(trainer_stats, "metrics", {}) or {}
    train_steps = getattr(trainer_stats, "global_step", None)
    final_loss = metrics.get("train_loss")

    print(f"Train steps: {train_steps if train_steps is not None else 'N/A'}")
    if final_loss is not None:
        print(f"Final loss: {final_loss:.4f}")
    else:
        print("Final loss: N/A")
    print("Raw trainer stats:", trainer_stats)
except NameError as exc:
    missing_name = str(exc)
    print("Training failed because a required variable is missing.")
    print("Most likely cause: the notebook cells were not run in order, or the runtime was restarted.")
    print("Please run Cell 1 through Cell 8 first, then rerun Cell 9.")
    raise
except Exception as exc:
    print("Training failed with an unexpected error:")
    traceback.print_exc()
    raise


In [ ]:
# Cell 10: Save LoRA Adapter
from pathlib import Path
import traceback

lora_dir = "/content/gemma4-router-lora"
try:
    model.save_pretrained(lora_dir)
    tokenizer.save_pretrained(lora_dir)
    if not Path(lora_dir).exists():
        raise FileNotFoundError(f"LoRA output directory was not created: {lora_dir}")
    print(f"LoRA adapter saved to {lora_dir}")
except NameError:
    print("Save failed because model/tokenizer is missing. Run Cells 1-9 first.")
    raise
except Exception:
    print("Save LoRA adapter failed:")
    traceback.print_exc()
    raise


In [ ]:
# Cell 11: Export to GGUF (Q4_K_M Quantization)
from pathlib import Path
import traceback

gguf_dir = "/content/gemma4-router-gguf"
gguf_path = f"{gguf_dir}/gguf-model-q4_k_m.gguf"
try:
    model.save_pretrained_gguf(
        gguf_dir,
        tokenizer,
        quantization_method="q4_k_m",
    )
    if not Path(gguf_path).exists():
        raise FileNotFoundError(f"Expected GGUF file not found: {gguf_path}")
    print("GGUF export complete!")
    print(f"Model: {gguf_path}")
except NameError:
    print("GGUF export failed because model/tokenizer is missing. Run Cells 1-10 first.")
    raise
except Exception:
    print("GGUF export failed:")
    traceback.print_exc()
    raise


In [ ]:
# Cell 12: Upload to Hugging Face Hub
import json
import os
import subprocess
import traceback
from datetime import datetime
from pathlib import Path
from google.colab import userdata
from huggingface_hub import HfApi

try:
    HF_TOKEN = userdata.get("HF_TOKEN")
    if not HF_TOKEN:
        raise ValueError("HF_TOKEN chưa được cấu hình trong Colab Secrets")

    api = HfApi()
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    repo_id = "diepxuan/dx-gemma4"

    # Upload GGUF model
    gguf_path = Path("/content/gemma4-router-gguf/gguf-model-q4_k_m.gguf")
    if not gguf_path.exists():
        raise FileNotFoundError(f"GGUF file not found: {gguf_path}. Run Cell 11 first.")
    api.upload_file(
        path_or_fileobj=str(gguf_path),
        path_in_repo=f"models/{timestamp}/gguf-model-q4_k_m.gguf",
        repo_id=repo_id,
        repo_type="model",
        commit_message=f"model {timestamp}",
        token=HF_TOKEN,
    )
    print(f"Uploaded GGUF: {gguf_path}")

    # Upload LoRA adapter (tar.gz)
    lora_dir = Path("/content/gemma4-router-lora")
    if not lora_dir.exists():
        raise FileNotFoundError(f"LoRA directory not found: {lora_dir}. Run Cell 10 first.")
    lora_archive = Path("/content/gemma4-router-lora.tar.gz")
    subprocess.run(
        ["tar", "-czf", str(lora_archive), "-C", "/content", "gemma4-router-lora"],
        check=True,
    )
    api.upload_file(
        path_or_fileobj=str(lora_archive),
        path_in_repo=f"models/{timestamp}/lora.tar.gz",
        repo_id=repo_id,
        repo_type="model",
        commit_message=f"lora adapter {timestamp}",
        token=HF_TOKEN,
    )
    print("Uploaded LoRA adapter")

    # Upload manifest
    train_metrics = getattr(trainer_stats, "metrics", {}) if "trainer_stats" in globals() else {}
    manifest = {
        "timestamp": timestamp,
        "model": "gemma-4-e2b",
        "quantization": "q4_k_m",
        "epochs": 3,
        "lora_rank": 16,
        "dataset_size": len(train_data) if "train_data" in globals() else None,
        "train_steps": getattr(trainer_stats, "global_step", None) if "trainer_stats" in globals() else None,
        "final_loss": train_metrics.get("train_loss") if train_metrics else None,
        "hf_repo": repo_id,
    }
    manifest_path = Path("/content/manifest.json")
    manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

    api.upload_file(
        path_or_fileobj=str(manifest_path),
        path_in_repo=f"models/{timestamp}/manifest.json",
        repo_id=repo_id,
        repo_type="model",
        commit_message=f"manifest {timestamp}",
        token=HF_TOKEN,
    )
    print("Uploaded manifest")
    print(f"\nAll files uploaded to: https://huggingface.co/{repo_id}/tree/main/models/{timestamp}")
except Exception:
    print("Upload failed:")
    traceback.print_exc()
    raise


In [ ]:
# Cell 13: Quick Inference Test
from unsloth.chat_templates import get_chat_template

test_prompts = [
    "Sếp cần viết API REST cho user management bằng FastAPI",
    "Lỗi TypeError khi chạy migration, anh fix giúp",
    "Tìm hiểu về vector databases, anh cần tổng quan chi tiết",
]

FastLanguageModel.for_inference(model)

print("Testing inference...\n")
for prompt in test_prompts:
    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    
    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.95,
        do_sample=True,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Input: {prompt}")
    print(f"Output: {response}")
    print("-" * 60)

## Done!

**Model uploaded to:** https://huggingface.co/diepxuan/dx-gemma4
**GGUF file:** `models/<timestamp>/gguf-model-q4_k_m.gguf`

Download model trên local server:
```bash
python3 ~/gemma4-router-finetune/pipeline.py --step download
```